<center><img src="../../image/dice.png" alt="DICE" style="width: 300px;"/></center>

# 14.OpenCV 摄像头机械臂物体追踪

@－－－－DICE－－－－<br>
@  文件名：14.OpenCV摄像头机械臂物体追踪.ipynb <br>
@  版本：V2.0 <br>
@  说明：OpenCV摄像头机械臂物体追踪<br>

### 调整机械臂到达Home 点，便于获取图像

In [1]:
import math
from arm5dof_uservo import Arm5DoFUServo
from config import *

### 机械臂创建

In [2]:
arm = Arm5DoFUServo()

### 到达Home点

In [3]:
# 回到home点
arm.home()

In [4]:
#到达拍摄位置
arm.uservo.set_servo_angle(0,88,interval=100)
arm.uservo.set_servo_angle(3,-80,interval=100)
arm.uservo.wait()

In [5]:
# 载入必要的库
import cv2
import numpy as np
import struct
import time

In [6]:
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display
from jetcam.utils import bgr8_to_jpeg

FGmaskComp_img = widgets.Image(format='jpeg', width=640, height=320)
frame_img = widgets.Image(format='jpeg', width=640, height=320)

dispaly_img = widgets.HBox([FGmaskComp_img,frame_img])
display(dispaly_img)

In [7]:
# 线程函数操作库
import threading # 线程
import ctypes
import inspect

# 线程结束代码
def _async_raise(tid, exctype):
    tid = ctypes.c_long(tid)
    if not inspect.isclass(exctype):
        exctype = type(exctype)
    res = ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, ctypes.py_object(exctype))
    if res == 0:
        raise ValueError("invalid thread id")
    elif res != 1:
        ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, None)
        raise SystemError("PyThreadState_SetAsyncExc failed")
        
def stop_thread(thread):
    _async_raise(thread.ident, SystemExit)

In [8]:
hueLower = widgets.IntSlider(min=0,max=255,step=1,description='hueLower：',value=96)
hueUpper = widgets.IntSlider(min=0,max=255,step=1,description='hueUpper：',value=120)

hue2Lower = widgets.IntSlider(min=0,max=255,step=1,description='hue2Lower：',value=50)
hue2Upper = widgets.IntSlider(min=0,max=255,step=1,description='hue2Upper：',value=0)

satLow = widgets.IntSlider(min=0,max=255,step=1,description='satLow：',value=157)
satHigh = widgets.IntSlider(min=0,max=255,step=1,description='satHigh：',value=255)

valLow = widgets.IntSlider(min=0,max=255,step=1,description='valLow：',value=100)
valHigh = widgets.IntSlider(min=0,max=255,step=1,description='valHigh：',value=255)

slider_img=widgets.VBox([
              widgets.HBox([hueLower,hueUpper]),
              widgets.HBox([hue2Lower,hue2Upper]),
              widgets.HBox([satLow,satHigh]),
              widgets.HBox([valLow,valHigh])
             ])
display(slider_img)

In [9]:
pose1_servo_raw_angle = arm.get_servo_angle_list()
# 打印舵机原始角度
j1_0, j2_n90, j3_90, j4_n90, j5_0, j6_90 = pose1_servo_raw_angle
print(f"J1: {j1_0:.1f} J2: {j2_n90:.1f} J3: {j3_90:.1f} J4: {j4_n90:.1f} J5: {j5_0:.1f} J6: {j6_90:.1f}")    

pan_angle =  j1_0
tilt_angle = j3_90

J1: 88.5 J2: -19.6 J3: -16.8 J4: -81.7 J5: 0.6 J6: 30.5


In [10]:
def Video_display():
    global pan_angle
    global tilt_angle
    
    capture = cv2.VideoCapture(0)

    width=640
    height=480
    while True: 
        ret, cframe_data = capture.read()
        if not ret:
            return ret, cframe_data
        cframe_data=cframe_data.copy()
        hsv=cv2.cvtColor(cframe_data,cv2.COLOR_BGR2HSV)

        hueLow=hueLower.value 
        hueUp=hueUpper.value

        hue2Low=hue2Lower.value
        hue2Up=hue2Upper.value

        Ls= satLow.value
        Us = satHigh.value

        Lv=valLow.value
        Uv=valHigh.value

        l_b=np.array([hueLow,Ls,Lv])
        u_b=np.array([hueUp,Us,Uv])

        l_b2=np.array([hue2Low,Ls,Lv])
        u_b2=np.array([hue2Up,Us,Uv])

        FGmask=cv2.inRange(hsv,l_b,u_b)
        FGmask2=cv2.inRange(hsv,l_b2,u_b2)
        FGmaskComp=cv2.add(FGmask,FGmask2)
        FGmaskComp_img.value = bgr8_to_jpeg(FGmaskComp)

        contours,_=cv2.findContours(FGmaskComp,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        contours=sorted(contours,key=lambda x:cv2.contourArea(x),reverse=True)
        
        for cnt in contours:
            area=cv2.contourArea(cnt)
            (x,y,w,h)=cv2.boundingRect(cnt)
            
            if area>=50:
                cv2.rectangle(cframe_data,(x,y),(x+w,y+h),(255,0,0),3)
                objX=x+w/2
                objY=y+h/2
                
                errorPan=objX-width/2
                errorTilt=objY-height/2            

                if abs(errorPan)>15:
                    pan_angle=pan_angle+errorPan/75    

                if abs(errorTilt)>15:
                    tilt_angle=tilt_angle-errorTilt/75
                if pan_angle>90.0:
                    pan_angle=90.0
                    print("Pan Out of  Range")           
                if pan_angle<-90.0:
                    pan_angle=-90.0
                    print("Pan Out of  Range")              

                if tilt_angle>90.0:
                    tilt_angle=90.0
                    print("Tilt Out of  Range") 
                if tilt_angle<-90.0:
                    tilt_angle=-90.0
                    print("Tilt Out of  Range")
               
                arm.uservo.set_servo_angle(0,pan_angle,interval=0)
                arm.uservo.set_servo_angle(2,tilt_angle,interval=0) 
                break 
        frame_img.value = bgr8_to_jpeg(cframe_data)
    capture.release()

### 启动机械臂追踪及启动摄像头处理

In [11]:
t1 = threading.Thread(target=Video_display)
t1.setDaemon(True)
t1.start()

In [12]:
# 结束线程
stop_thread(t1)